In [ ]:
import io, os, zipfile, urllib.request, argparse, time
import numpy as np
import scipy.sparse as sp
import tensorflow as tf

REPO_ZIP_URL = "https://codeload.github.com/xiangwang1223/neural_graph_collaborative_filtering/zip/refs/heads/master"
DATASETS = ("gowalla",)

def download_and_extract(data_root="./data", force=False):
    os.makedirs(data_root, exist_ok=True)
    already_have_all = all(
        os.path.exists(os.path.join(data_root, ds, "train.txt"))
        and os.path.exists(os.path.join(data_root, ds, "test.txt")) for ds in DATASETS
    )
    if already_have_all and not force: return
    req = urllib.request.Request(REPO_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as resp: zip_bytes = resp.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        top = zf.namelist()[0].split("/")[0]
        for ds in DATASETS:
            for fname in ("train.txt", "test.txt"):
                member = f"{top}/Data/{ds}/{fname}"
                out_path = os.path.join(data_root, ds, fname)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                with zf.open(member) as src, open(out_path, "wb") as dst: dst.write(src.read())

class Dataset:
    def __init__(self, name, data_root="./data", sample_users=None, sample_items=None, seed=2019, split_ratio=None):
        assert name in DATASETS
        self.name, self.rng = name, np.random.RandomState(seed)
        train_path, test_path = os.path.join(data_root, name, "train.txt"), os.path.join(data_root, name, "test.txt")

        def _read_file_internal(path):
            user_dict, max_u, max_i = {}, -1, -1
            with open(path, "r") as f:
                for line in f:
                    parts = line.strip().split(" ")
                    if len(parts) < 1 or parts[0] == "": continue
                    u = int(parts[0]); items = [int(x) for x in parts[1:] if x != ""]
                    user_dict[u] = items; max_u = max(max_u, u); max_i = max(max_i, max(items)) if items else max_i
            return user_dict, max_u, max_i

        if split_ratio is None:
            self.train_user_dict, n_users_tr, n_items_tr = _read_file_internal(train_path)
            self.test_user_dict, n_users_te, n_items_te = _read_file_internal(test_path)
            self.n_users = max(n_users_tr, n_users_te) + 1; self.n_items = max(n_items_tr, n_items_te) + 1
        else:
            combined_user_dict, max_u, max_i = {}, -1, -1 # Initialize max_u and max_i here
            for path in [train_path, test_path]:
                ud, mu, mi = _read_file_internal(path)
                for u, items in ud.items(): combined_user_dict.setdefault(u, []).extend(items)
                max_u, max_i = max(max_u, mu), max(max_i, mi)
            self.n_users, self.n_items = max_u + 1, max_i + 1
            all_interactions = [(u, i) for u, items in combined_user_dict.items() for i in items]
            self.rng.shuffle(all_interactions)
            train_size = int(len(all_interactions) * split_ratio)
            train_interactions_list, test_interactions_list = all_interactions[:train_size], all_interactions[train_size:]
            self.train_user_dict, self.test_user_dict = {}, {}
            for u, i in train_interactions_list: self.train_user_dict.setdefault(u, []).append(i)
            for u, i in test_interactions_list: self.test_user_dict.setdefault(u, []).append(i)

        # Optional sub-sampling, purely as a fast smoke-test knob for
        # people who want to sanity-check the pipeline before committing
        # to a multi-hour full run. Leave both as None for the real thing.
        if sample_users is not None or sample_items is not None:
            self._subsample(sample_users, sample_items)

        self.train_pairs = np.array([(u, i) for u, items in self.train_user_dict.items() for i in items], dtype=np.int64)
        self.n_train_interactions = len(self.train_pairs)
        rows, cols = self.train_pairs[:, 0], self.train_pairs[:, 1]
        self.R = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(self.n_users, self.n_items))

    def _subsample(self, sample_users, sample_items):
        """Debug-only: restrict to a random subset of users/items so the
        whole pipeline can be smoke-tested in seconds instead of hours."""
        n_u = min(sample_users or self.n_users, self.n_users)
        n_i = min(sample_items or self.n_items, self.n_items)
        keep_users = set(self.rng.choice(self.n_users, size=n_u, replace=False).tolist())
        keep_items = set(self.rng.choice(self.n_items, size=n_i, replace=False).tolist())

        def _filter(d):
            out = {}
            for u, items in d.items():
                if u not in keep_users:
                    continue
                filt = [i for i in items if i in keep_items]
                out[u] = filt
            return out

        self.train_user_dict = _filter(self.train_user_dict)
        self.test_user_dict = _filter(self.test_user_dict)
        # remap ids to a dense [0, n) range
        u_map = {u: idx for idx, u in enumerate(sorted(keep_users))}
        i_map = {i: idx for idx, i in enumerate(sorted(keep_items))}
        self.train_user_dict = {u_map[u]: [i_map[i] for i in items]
                                 for u, items in self.train_user_dict.items()}
        self.test_user_dict = {u_map[u]: [i_map[i] for i in items]
                                for u, items in self.test_user_dict.items()}
        self.n_users, self.n_items = len(u_map), len(i_map)

    def sample_bpr_batch(self, batch_size):
        idx = self.rng.randint(0, self.n_train_interactions, size=batch_size)
        users, pos_items, neg_items = self.train_pairs[idx, 0], self.train_pairs[idx, 1], np.empty(batch_size, dtype=np.int64)
        for k, u in enumerate(users):
            pos_set = self.train_user_dict[u]
            while True:
                j = self.rng.randint(0, self.n_items)
                if j not in pos_set: neg_items[k] = j; break
        return users.astype(np.int32), pos_items.astype(np.int32), neg_items.astype(np.int32)

    def get_norm_laplacian(self):
        n = self.n_users + self.n_items; R = self.R.tocsr()
        zero_uu = sp.csr_matrix((self.n_users, self.n_users), dtype=np.float32)
        zero_ii = sp.csr_matrix((self.n_items, self.n_items), dtype=np.float32)
        A = sp.bmat([[zero_uu, R], [R.T, zero_ii]], format="csr", dtype=np.float32)
        degree = np.array(A.sum(axis=1)).flatten(); degree[degree == 0] = 1.0; d_inv_sqrt = np.power(degree, -0.5)
        D_inv_sqrt = sp.diags(d_inv_sqrt)
        L = D_inv_sqrt.dot(A).dot(D_inv_sqrt).tocoo(); L_plus_I = (L + sp.eye(n, format="coo")).tocoo()
        return L.astype(np.float32), L_plus_I.astype(np.float32)

def _sparse_to_tf(coo): return tf.sparse.SparseTensor(np.stack([coo.row, coo.col], axis=1).astype(np.int64), coo.data.astype(np.float32), coo.shape)

class MF(tf.keras.Model):
    def __init__(self, n_users, n_items, embed_size=64, seed=2019):
        super().__init__()
        init = tf.keras.initializers.GlorotUniform(seed=seed)
        self.user_emb = self.add_weight(shape=(n_users, embed_size), initializer=init, trainable=True, name="mf_user_emb")
        self.item_emb = self.add_weight(shape=(n_items, embed_size), initializer=init, trainable=True, name="mf_item_emb")

    def bpr_forward(self, u, i, j, training=False):
        eu, ei, ej = tf.gather(self.user_emb, u), tf.gather(self.item_emb, i), tf.gather(self.item_emb, j)
        pos, neg = tf.reduce_sum(eu * ei, axis=1), tf.reduce_sum(eu * ej, axis=1)
        reg = tf.nn.l2_loss(eu) + tf.nn.l2_loss(ei) + tf.nn.l2_loss(ej); return pos, neg, reg

    def full_scores(self, user_batch, item_chunk=None): return tf.matmul(tf.gather(self.user_emb, user_batch), self.item_emb, transpose_b=True)

class NeuMF(tf.keras.Model):
    def __init__(self, n_users, n_items, embed_size=64, seed=2019):
        super().__init__()
        init = tf.keras.initializers.GlorotUniform(seed=seed)
        self.gmf_user_emb = self.add_weight(shape=(n_users, embed_size), initializer=init, trainable=True, name="gmf_user_emb")
        self.gmf_item_emb = self.add_weight(shape=(n_items, embed_size), initializer=init, trainable=True, name="gmf_item_emb")
        self.mlp_user_emb = self.add_weight(shape=(n_users, embed_size), initializer=init, trainable=True, name="mlp_user_emb")
        self.mlp_item_emb = self.add_weight(shape=(n_items, embed_size), initializer=init, trainable=True, name="mlp_item_emb")
        mlp_dim = 2 * embed_size
        self.mlp_layer1 = tf.keras.layers.Dense(mlp_dim, activation="relu", name="mlp_l1"); self.mlp_layer1.build((None, mlp_dim))
        self.mlp_layer2 = tf.keras.layers.Dense(mlp_dim, activation="relu", name="mlp_l2"); self.mlp_layer2.build((None, mlp_dim))
        self.out_layer = tf.keras.layers.Dense(1, use_bias=False, name="neumf_out"); self.out_layer.build((None, embed_size + mlp_dim))

    def _forward(self, eu_gmf, ei_gmf, eu_mlp, ei_mlp):
        gmf_vec = eu_gmf * ei_gmf; mlp_in = tf.concat([eu_mlp, ei_mlp], axis=-1)
        mlp_vec = self.mlp_layer2(self.mlp_layer1(mlp_in)); joint = tf.concat([gmf_vec, mlp_vec], axis=-1)
        return tf.squeeze(self.out_layer(joint), axis=-1)

    def bpr_forward(self, u, i, j, training=False):
        eu_g, ei_g, ej_g = tf.gather(self.gmf_user_emb, u), tf.gather(self.gmf_item_emb, i), tf.gather(self.gmf_item_emb, j)
        eu_m, ei_m, ej_m = tf.gather(self.mlp_user_emb, u), tf.gather(self.mlp_item_emb, i), tf.gather(self.mlp_item_emb, j)
        pos, neg = self._forward(eu_g, ei_g, eu_m, ei_m), self._forward(eu_g, ej_g, eu_m, ej_m)
        reg = (tf.nn.l2_loss(eu_g) + tf.nn.l2_loss(ei_g) + tf.nn.l2_loss(ej_g) + tf.nn.l2_loss(eu_m) + tf.nn.l2_loss(ei_m) + tf.nn.l2_loss(ej_m)); return pos, neg, reg

    def full_scores(self, user_batch, item_chunk=4096):
        n_items = self.gmf_item_emb.shape[0]; b = len(user_batch); eu_g, eu_m = tf.gather(self.gmf_user_emb, user_batch), tf.gather(self.mlp_user_emb, user_batch)
        all_scores = []
        for start in range(0, n_items, item_chunk):
            end = min(start + item_chunk, n_items); c = end - start; ei_g, ei_m = self.gmf_item_emb[start:end], self.mlp_item_emb[start:end]
            eu_g_rep, ei_g_rep = tf.repeat(eu_g, repeats=c, axis=0), tf.tile(ei_g, [b, 1])
            eu_m_rep, ei_m_rep = tf.repeat(eu_m, repeats=c, axis=0), tf.tile(ei_m, [b, 1])
            all_scores.append(tf.reshape(self._forward(eu_g_rep, ei_g_rep, eu_m_rep, ei_m_rep), [b, c]))
        return tf.concat(all_scores, axis=1)

class LightGCN(tf.keras.Model):
    def __init__(self, n_users, n_items, norm_lap, embed_size=64, n_layers=3, node_dropout=0.0, mess_dropout=0.1, seed=2019):
        super().__init__()
        self.n_users, self.n_items, self.node_dropout, self.mess_dropout = n_users, n_items, node_dropout, mess_dropout
        self.n_layers = n_layers
        init = tf.keras.initializers.GlorotUniform(seed=seed)
        self.ego_emb = self.add_weight(shape=(n_users + n_items, embed_size), initializer=init, trainable=True, name="lightgcn_ego_emb")
        self.L = _sparse_to_tf(norm_lap.tocoo())

    def _dropout_sparse(self, sp_mat, keep_prob):
        if keep_prob >= 1.0: return sp_mat
        random_mask = tf.random.uniform([sp_mat.values.shape[0]]) < keep_prob
        idx, val = tf.boolean_mask(sp_mat.indices, random_mask), tf.boolean_mask(sp_mat.values, random_mask) / keep_prob
        return tf.sparse.SparseTensor(idx, val, sp_mat.dense_shape)

    def propagate(self, training=False):
        L = self._dropout_sparse(self.L, 1.0 - self.node_dropout) if training and self.node_dropout > 0 else self.L
        E, all_layers = self.ego_emb, [self.ego_emb]
        for l in range(self.n_layers):
            E = tf.sparse.sparse_dense_matmul(L, E)
            E = tf.nn.dropout(E, rate=self.mess_dropout) if training and self.mess_dropout > 0 else E
            E = tf.math.l2_normalize(E, axis=1); all_layers.append(E)
        return tf.concat(all_layers, axis=1)

    def bpr_forward(self, u, i, j, training=False):
        E = self.propagate(training=training) # Corrected line
        eu, ei, ej = tf.gather(E[:self.n_users], u), tf.gather(E[self.n_users:], i), tf.gather(E[self.n_items:], j)
        pos, neg = tf.reduce_sum(eu * ei, axis=1), tf.reduce_sum(eu * ej, axis=1)
        reg = tf.nn.l2_loss(eu) + tf.nn.l2_loss(ei) + tf.nn.l2_loss(ej); return pos, neg, reg

    def full_scores(self, user_batch, item_chunk=None, E=None):
        E = self.propagate(training=False) if E is None else E
        return tf.matmul(tf.gather(E[:self.n_users], user_batch), E[self.n_users:], transpose_b=True)

def _dcg_at_k(hits, k):
    hits = np.asarray(hits, dtype=np.float64)[:k]; return 0.0 if hits.size == 0 else float(np.sum(hits * (1.0 / np.log2(np.arange(2, hits.size + 2))))) # noqa

def _ndcg_at_k(ranked_items, test_items_set, k):
    dcg = _dcg_at_k([1.0 if it in test_items_set else 0.0 for it in ranked_items[:k]], k)
    idcg = _dcg_at_k([1.0] * min(len(test_items_set), k), k); return dcg / idcg if idcg > 0 else 0.0

def _recall_at_k(ranked_items, test_items_set, k):
    return None if len(test_items_set) == 0 else sum(1 for it in ranked_items[:k] if it in test_items_set) / len(test_items_set)

def evaluate_model(model, dataset, k=20, user_batch_size=256, item_chunk=4096, full_scores_kwargs=None):
    full_scores_kwargs = full_scores_kwargs or {}
    test_users = [u for u, items in dataset.test_user_dict.items() if len(items) > 0]; recalls, ndcgs = [], []
    for start in range(0, len(test_users), user_batch_size):
        batch_users = test_users[start:start + user_batch_size]
        scores = model.full_scores(np.array(batch_users, dtype=np.int32), item_chunk=item_chunk, **full_scores_kwargs)
        scores = scores.numpy() if isinstance(scores, tf.Tensor) else np.asarray(scores)
        for row, u in enumerate(batch_users):
            train_items = dataset.train_user_dict.get(u, [])
            if train_items: scores[row, train_items] = -np.inf

        for row, u in enumerate(batch_users):
            s = scores[row]; top_idx = np.argpartition(-s, k)[:k] if s.shape[0] > k else np.arange(s.shape[0])
            top_idx = top_idx[np.argsort(-s[top_idx])]
            test_set = set(dataset.test_user_dict[u])
            r, n = _recall_at_k(top_idx, test_set, k), _ndcg_at_k(top_idx, test_set, k)
            if r is not None: recalls.append(r); ndcgs.append(n)
    return float(np.mean(recalls)), float(np.mean(ndcgs))

MODEL_REGISTRY = {"MF": MF, "NeuMF": NeuMF, "LightGCN-3": LightGCN}

def build_model(name, ds, args):
    if name == "MF": return MF(ds.n_users, ds.n_items, embed_size=args.embed_size, seed=args.seed)
    if name == "NeuMF": return NeuMF(ds.n_users, ds.n_items, embed_size=args.embed_size, seed=args.seed)
    if name == "LightGCN-3":
        L, _ = ds.get_norm_laplacian()
        return LightGCN(ds.n_users, ds.n_items, L, embed_size=args.embed_size, n_layers=args.ngcf_layers, node_dropout=args.node_dropout, mess_dropout=args.mess_dropout, seed=args.seed)

def make_train_step(model, optimizer, l2_reg, batch_size):
    @tf.function
    def step(u, i, j):
        with tf.GradientTape() as tape:
            pos, neg, reg = model.bpr_forward(u, i, j, training=True)
            loss = -tf.reduce_mean(tf.math.log_sigmoid(pos - neg)) + l2_reg * reg / tf.cast(batch_size, tf.float32)
        grads = tape.gradient(loss, model.trainable_variables); grads = [tf.clip_by_norm(g, 5.0) for g in grads]
        optimizer.apply_gradients(zip(grads, model.trainable_variables)); return loss
    return step

def train_model(name, model, ds, args):
    optimizer = tf.keras.optimizers.Adam(learning_rate=args.lr); steps_per_epoch = max(1, ds.n_train_interactions // args.batch_size)
    best_recall, best_ndcg, patience, train_step = -1.0, -1.0, 0, make_train_step(model, optimizer, args.l2_reg, args.batch_size)
    for epoch in range(1, args.epochs + 1):
        t0 = time.time(); epoch_loss = 0.0
        for _ in range(steps_per_epoch):
            u, i, j = ds.sample_bpr_batch(args.batch_size)
            epoch_loss += float(train_step(tf.constant(u), tf.constant(i), tf.constant(j)))
        epoch_loss /= steps_per_epoch
        do_eval = (epoch % args.eval_every == 0) or (epoch == args.epochs)
        msg = f"  [{name}] epoch {epoch:>4}/{args.epochs}  loss={epoch_loss:.4f}  ({time.time()-t0:.1f}s)"
        if do_eval:
            recall, ndcg = evaluate_model(model, ds, k=args.k, user_batch_size=args.eval_user_batch, item_chunk=args.eval_item_chunk)
            msg += f"  recall@{args.k}={recall:.4f}  ndcg@{args.k}={ndcg:.4f}"
            if recall > best_recall: best_recall, best_ndcg, patience = recall, ndcg, 0
            else: patience += 1
        print(msg)
        if args.early_stop > 0 and patience >= args.early_stop: print(f"  [{name}] early stopping"); break
    return (best_recall, best_ndcg) if best_recall >= 0 else evaluate_model(model, ds, k=args.k)

def main():
    p = argparse.ArgumentParser(formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--datasets", nargs='+', default=['gowalla'], choices=list(DATASETS)) # Modified: default to 'gowalla'
    p.add_argument("--data_root", default="./data")
    p.add_argument("--models", nargs='+', default=["MF", "NeuMF", "LightGCN-3"], choices=["MF", "NeuMF", "LightGCN-3"])
    p.add_argument("--embed_size", type=int, default=64)
    p.add_argument("--ngcf_layers", type=int, default=3, help="number of LightGCN propagation layers (L=3 -> LightGCN-3)")
    p.add_argument("--batch_size", type=int, default=1024)
    p.add_argument("--epochs", type=int, default=10)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--l2_reg", type=float, default=1e-5)
    p.add_argument("--node_dropout", type=float, default=0.0)
    p.add_argument("--mess_dropout", type=float, default=0.1)
    p.add_argument("--k", type=int, default=20)
    p.add_argument("--eval_every", type=int, default=10)
    p.add_argument("--eval_user_batch", type=int, default=256)
    p.add_argument("--eval_item_chunk", type=int, default=4096)
    p.add_argument("--early_stop", type=int, default=5, help="stop after this many evals with no recall@K improvement (0 = disabled)")
    p.add_argument("--seed", type=int, default=2019)
    p.add_argument("--sample_users", type=int, default=None)
    p.add_argument("--sample_items", type=int, default=None)
    p.add_argument("--split_ratio", type=float, default=0.8, help="Ratio for train/test split (e.g., 0.8 for 80/20). If None, uses original LOO split.")
    args, unknown = p.parse_known_args()

    np.random.seed(args.seed)
    tf.random.set_seed(args.seed)

    download_and_extract(args.data_root)

    all_results = {}  # {dataset: {model: (recall, ndcg)}}
    for ds_name in args.datasets:
        print(f"\n{'='*70}\nDataset: {ds_name}\n{'='*70}")
        ds = Dataset(ds_name, data_root=args.data_root,
                     sample_users=args.sample_users, sample_items=args.sample_items,
                     seed=args.seed, split_ratio=args.split_ratio)
        all_results[ds_name] = {}
        for model_name in args.models:
            print(f"\n--- Training {model_name} on {ds_name} ---")
            model = build_model(model_name, ds, args)
            recall, ndcg = train_model(model_name, model, ds, args)
            all_results[ds_name][model_name] = (recall, ndcg)

    print_table2(all_results, args)

def print_table2(all_results, args):
    datasets = list(all_results.keys())
    models = args.models
    col_w = 22

    print(f"\n\n{'='*70}\nTable 2 (reproduced): Overall Performance Comparison\n{'='*70}")
    header = f"{'':10s}" + "".join(f"{ds:>{col_w}s}" for ds in datasets)
    print(header)
    sub = f"{'':10s}" + "".join(f"{(f'recall@{str(args.k)}'):>11s}{(f'ndcg@{str(args.k)}'):>11s}" for _ in datasets)
    print(sub)
    print("-" * len(sub))

    for m in models:
        row = f"{m:10s}"
        for ds in datasets:
            r, n = all_results[ds][m]
            row += f"{r:>11.4f}{n:>11.4f}"
        print(row)

    if "LightGCN-3" in models: # Updated from NGCF-3
        base_models = [m for m in models if m != "LightGCN-3"]
        if base_models:
            print("-" * len(sub))
            row = f"{'%Improv.':10s}"
            for ds in datasets:
                lightgcn_r, lightgcn_n = all_results[ds]["LightGCN-3"]
                best_r = max(all_results[ds][m][0] for m in base_models)
                best_n = max(all_results[ds][m][1] for m in base_models)
                imp_r = 100 * (lightgcn_r - best_r) / best_r if best_r > 0 else float("nan")
                imp_n = 100 * (lightgcn_n - best_n) / best_n if best_n > 0 else float("nan")
                row += f"{imp_r:>10.2f}%{imp_n:>10.2f}%"
            print(row)


if __name__ == "__main__":
    main()


Dataset: gowalla

--- Training MF on gowalla ---
  [MF] epoch    1/10  loss=0.6583  (11.3s)
  [MF] epoch    2/10  loss=0.4100  (7.2s)
  [MF] epoch    3/10  loss=0.2502  (7.2s)
  [MF] epoch    4/10  loss=0.1761  (7.1s)
  [MF] epoch    5/10  loss=0.1354  (7.3s)
  [MF] epoch    6/10  loss=0.1099  (7.3s)
  [MF] epoch    7/10  loss=0.0932  (7.4s)
  [MF] epoch    8/10  loss=0.0813  (7.4s)
  [MF] epoch    9/10  loss=0.0724  (7.2s)
  [MF] epoch   10/10  loss=0.0659  (7.2s)  recall@20=0.1211  ndcg@20=0.0952

--- Training NeuMF on gowalla ---
  [NeuMF] epoch    1/10  loss=0.5419  (12.6s)
  [NeuMF] epoch    2/10  loss=0.2968  (11.6s)
  [NeuMF] epoch    3/10  loss=0.1253  (10.6s)
  [NeuMF] epoch    4/10  loss=0.0757  (10.6s)
  [NeuMF] epoch    5/10  loss=0.0561  (10.6s)
  [NeuMF] epoch    6/10  loss=0.0450  (10.7s)
  [NeuMF] epoch    7/10  loss=0.0390  (10.8s)
  [NeuMF] epoch    8/10  loss=0.0341  (10.6s)
  [NeuMF] epoch    9/10  loss=0.0309  (10.6s)
  [NeuMF] epoch   10/10  loss=0.0279  (10.7s) 